In [ ]:
import pandas as pd
import torch
from sklearn.manifold import TSNE
import plotly.express as px
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
from ddi_graph_neural_network.train_model import main
from ddi_graph_neural_network.config import Config

/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Visualize learned embeddings

In [ ]:
def get_node_embeddings(model, data):
    model.eval()
    with torch.no_grad():
        node_embeddings = model.encode(data.x, data.edge_index)
    node_embeddings = node_embeddings.cpu().numpy()
    return node_embeddings


def get_reduced_embeddings(node_embeddings):
    # Initialize t-SNE with 2 components for 2D visualization
    tsne = TSNE(n_components=2)  # , random_state=42) # 42

    # Apply t-SNE to the drug embeddings
    reduced_embeddings = tsne.fit_transform(node_embeddings)

    return pd.DataFrame(reduced_embeddings, columns=["TSNE-1", "TSNE-2"])


def plot_embeddings(reduced_embeddings_df):
    # Create a scatter plot using Plotly
    fig = px.scatter(
        reduced_embeddings_df,
        x="TSNE-1",
        y="TSNE-2",
        # use hover_data to show drug IDs
        hover_data=[reduced_embeddings_df.index],
        title="t-SNE Visualization of Drug Embeddings",
        width=800,
        height=600,
    )
    fig.show()
    # return fig

In [ ]:
# globals().update(main()) # Pass local variables from main() to the global scope

In [5]:
runs_neg = main(Config(seed=42, current_graph="CRESCENDDI", take_negative_samples=True))
runs_pos = main(Config(seed=42, current_graph="CRESCENDDI", take_negative_samples=False))

Current graph: CRESCENDDI
======== DESC_GPT ========
-------------------------------
-- FINAL RESULTS FOR GRAPH CRESCENDDI | FEATURE DESC_GPT -- 
Graph Data:  CRESCENDDI
ROC_AUC: 0.8386
PR_AUC: 0.8499
Current graph: CRESCENDDI
======== DESC_GPT ========
-------------------------------
-- FINAL RESULTS FOR GRAPH CRESCENDDI | FEATURE DESC_GPT -- 
Graph Data:  CRESCENDDI
ROC_AUC: 0.8356
PR_AUC: 0.8336


In [ ]:
# feature_type = 'GPT+Desc'
# lr = 0.0003
# key = (feature_type, lr)
vars = runs_neg


In [ ]:
graph_with_emb = vars["data"]
model = vars["model"]
node_id_map = vars["node_id_map"]

In [8]:
reversed_node_id_map = {v: k for k, v in node_id_map.items()}

In [9]:
embedding = get_node_embeddings(model, graph_with_emb)
embedding = get_reduced_embeddings(embedding)
embedding.index = embedding.index.map(lambda x: reversed_node_id_map.get(int(x), f"unknown_{int(x)}"))


In [10]:
plot_embeddings(embedding)

## Confusion matrix

In [ ]:
pd.DataFrame(vars["data"].edge_index.t().cpu().numpy(), columns=["source", "target"]).duplicated().sum()

0

In [ ]:
pd.DataFrame(vars["data"].edge_index.t().cpu().numpy(), columns=["source", "target"])

,source,target
0,0,5
1,0,36
2,0,85
3,0,108
4,0,136
...,...,...
7563,442,392
7564,442,393
7565,442,397
7566,442,411


In [ ]:
test_data = vars["test_data"]
test_scores = vars["test_scores"]

In [14]:
test_scores

array([0.6347578 , 0.58625966, 0.6044377 , 0.64834386, 0.74007463,
       0.59642935, 0.64786893, 0.53991586, 0.62968415, 0.55155754,
       0.68780106, 0.6421486 , 0.8181399 , 0.78024703, 0.7201788 ,
       0.7030553 , 0.72844183, 0.67383236, 0.5752992 , 0.74144197,
       0.5233105 , 0.7025262 , 0.5544258 , 0.6681964 , 0.713229  ,
       0.6451554 , 0.7585238 , 0.72519076, 0.71472114, 0.64960504,
       0.5365869 , 0.68680435, 0.725549  , 0.59630495, 0.5739732 ,
       0.6384399 , 0.7217093 , 0.7947069 , 0.6046086 , 0.7350466 ,
       0.86005956, 0.6647661 , 0.6785782 , 0.73115426, 0.5717995 ,
       0.6350676 , 0.53283346, 0.86187935, 0.7242761 , 0.64808184,
       0.55682176, 0.5111611 , 0.5881191 , 0.6943061 , 0.7111076 ,
       0.68314004, 0.65006703, 0.7598426 , 0.6378213 , 0.72957474,
       0.6874002 , 0.7874552 , 0.6251656 , 0.80240285, 0.75248903,
       0.65542454, 0.57041556, 0.70040584, 0.72811824, 0.67840916,
       0.6418629 , 0.7851956 , 0.76434314, 0.57239735, 0.72835

In [ ]:
# find optimal prediction threshold
def find_optimal_threshold(y_true, y_scores):
    from sklearn.metrics import precision_recall_curve, f1_score

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls)

    optimal_idx = f1_scores.argmax()
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold


y_true = test_data.edge_label.cpu().numpy()
optimal_threshold = find_optimal_threshold(y_true, test_scores)
print(f"Optimal threshold: {optimal_threshold}")

Optimal threshold: 0.5817369818687439


In [ ]:
# print confusion matrix at optimal threshold
def confusion_matrix_at_threshold(y_true, y_scores, optimal_threshold):
    from sklearn.metrics import confusion_matrix

    y_pred = (y_scores >= optimal_threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    cm_string = "[tn, fp]\n[fn, tp]\n"
    print("Confusion Matrix:")
    print(cm_string)
    print(cm)


confusion_matrix_at_threshold(y_true, test_scores, optimal_threshold)


def get_f1_at_threshold(y_true, y_scores, threshold):
    from sklearn.metrics import f1_score

    y_pred = (y_scores >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred)
    return f1


f1_at_optimal = get_f1_at_threshold(y_true, test_scores, optimal_threshold)
print(f"F1 Score at optimal threshold: {f1_at_optimal}")

Confusion Matrix:
[tn, fp]
[fn, tp]

[[285 163]
 [ 69 392]]
F1 Score at optimal threshold: 0.7716535433070866


In [ ]:
import numpy as np
from sklearn.metrics import f1_score


def compute_per_drug_f1_scores(test_data, test_scores, optimal_threshold, reversed_node_id_map):
    edge_idx = test_data.edge_label_index.cpu().numpy()  # shape (2, E)
    y_true = test_data.edge_label.cpu().numpy().astype(int)  # shape (E,)
    y_scores = np.asarray(test_scores)
    thr = float(optimal_threshold)

    src, dst = edge_idx[0], edge_idx[1]
    all_nodes = np.unique(np.concatenate([src, dst]))

    rows = []
    for node in all_nodes:
        drug_id = reversed_node_id_map.get(int(node))
        if drug_id is None:
            print(f"Warning: Node {node} not found in reversed_node_id_map.")
            continue
        mask = (src == node) | (dst == node)
        if not np.any(mask):
            print(f"Warning: No edges found for node {node} (drug_id: {drug_id}).")
            continue
        yt = y_true[mask]
        ys = y_scores[mask]
        yp = (ys >= thr).astype(int)
        f1 = f1_score(yt, yp, zero_division=0)
        rows.append({"drug_id": drug_id, "n_edges": int(mask.sum()), "f1": float(f1)})

    per_drug_f1 = pd.DataFrame(rows).sort_values(["f1", "n_edges"], ascending=[False, False])
    # display(per_drug_f1)
    return per_drug_f1


per_drug_f1 = compute_per_drug_f1_scores(test_data, test_scores, optimal_threshold, reversed_node_id_map)
per_drug_f1["index"] = per_drug_f1["drug_id"].map(
    lambda x: list(node_id_map.keys()).index(x) if x in node_id_map else -1
)
per_drug_f1.set_index("index", inplace=True)

In [18]:
per_drug_f1.head(10)

,drug_id,n_edges,f1
index,,,
387,DB06736,13,1.0
392,DB08439,13,1.0
415,DB09068,11,1.0
152,DB00620,10,1.0
352,DB01600,10,1.0
99,DB00477,9,1.0
204,DB00802,9,1.0
209,DB00814,9,1.0
266,DB01050,9,1.0


In [ ]:
DRUG_OF_INTEREST = "DB00749"
from sklearn.metrics import confusion_matrix


def confusion_matrix_for_drug(drug_id, test_data, test_scores, optimal_threshold, reversed_node_id_map):
    node_id = None
    for k, v in reversed_node_id_map.items():
        if v == drug_id:
            node_id = k
            break
    if node_id is None:
        print(f"Drug ID {drug_id} not found.")
        return

    edge_idx = test_data.edge_label_index.cpu().numpy()  # shape (2, E)
    y_true = test_data.edge_label.cpu().numpy().astype(int)  # shape (E,)
    y_scores = np.asarray(test_scores)
    thr = float(optimal_threshold)

    src, dst = edge_idx[0], edge_idx[1]
    mask = (src == node_id) | (dst == node_id)
    if not np.any(mask):
        print(f"No edges found for drug ID {drug_id}.")
        return

    yt = y_true[mask]
    ys = y_scores[mask]
    yp = (ys >= thr).astype(int)

    cm = confusion_matrix(yt, yp)
    cm_string = "[tn, fp]\n[fn, tp]\n"
    print(f"Confusion Matrix for drug ID {drug_id}:")
    print(cm_string)
    print(cm)


confusion_matrix_for_drug(DRUG_OF_INTEREST, test_data, test_scores, optimal_threshold, reversed_node_id_map)


def confusion_matrix_for_drug_for_all_possible_edges(drug_id, data, model, threshold, reversed_node_id_map):
    # map external ID -> internal idx
    node_id = None
    for k, v in reversed_node_id_map.items():
        if v == drug_id:
            node_id = k
            break
    if node_id is None:
        print(f"Drug ID {drug_id} not found.")
        return

    device = next(model.parameters()).device
    node_id_t = torch.tensor(node_id, dtype=torch.long, device=device)

    # Candidate edges: node_id to all others (exclude self)
    all_targets = torch.arange(data.num_nodes, device=device, dtype=torch.long)
    mask_not_self = all_targets != node_id_t
    all_targets = all_targets[mask_not_self]
    srcs = torch.full_like(all_targets, node_id_t)
    edge_index_candidate = torch.stack([srcs, all_targets], dim=0)  # shape [2, N-1]

    # Encode with the TRUE graph, not the candidate edges
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x.to(device), data.edge_index.to(device))
        logits = model.decode(z, edge_index_candidate).view(-1)
        global probs
        probs = logits.sigmoid().detach().cpu().numpy()

    # Build y_true: 1 if (node_id, t) is an edge in the TRUE graph (undirected)
    ei = data.edge_index.cpu().numpy()
    src, dst = ei[0], ei[1]
    # neighbors of node_id (treat as undirected)
    neighbors = set(dst[src == node_id].tolist()) | set(src[dst == node_id].tolist())
    targets_cpu = all_targets.cpu().numpy()
    y_true = np.array([1 if int(t) in neighbors else 0 for t in targets_cpu], dtype=int)

    # Predictions at threshold
    y_pred = (probs >= float(threshold)).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix for drug ID", drug_id, "(all possible edges):")
    print("[tn, fp]\n[fn, tp]\n")
    print(cm)


# Usage
confusion_matrix_for_drug_for_all_possible_edges(
    DRUG_OF_INTEREST, vars["data"], vars["model"], 0.96, reversed_node_id_map
)

Confusion Matrix for drug ID DB00749:
[tn, fp]
[fn, tp]

[[ 0  1]
 [ 0 11]]
Confusion Matrix for drug ID DB00749 (all possible edges):
[tn, fp]
[fn, tp]

[[ 97 263]
 [  3  80]]


probs
# histogram     of probs
import plotly.graph_objects as go
fig = go.Figure(data=[go.Histogram(x=probs, nbinsx=50)])
fig.update_layout(
    title="Histogram of Predicted Probabilities for All Possible Edges from Drug 'DB08932'",
    xaxis_title="Predicted Probability",
    yaxis_title="Count",
    width=800,
    height=500
)
fig.show()

## Use original embeddings

In [24]:
# use original embedding if disered
path = "/data/giobbi/embeddings/DESC_GPT.csv"
original_embedding = pd.read_csv(path, sep="\t", index_col=0)
original_embedding.set_index(original_embedding.columns[0], inplace=True)
original_embedding

# make tsne
original_embedding.select_dtypes(include=["float64", "float32"])
tsne = TSNE(n_components=2)
reduced_original_embedding = tsne.fit_transform(
    original_embedding.select_dtypes(include=["float64", "float32"])
)
reduced_original_embedding_df = pd.DataFrame(reduced_original_embedding, columns=["TSNE-1", "TSNE-2"])
reduced_original_embedding_df.index = original_embedding.index

# join original_embedding_df to embedding
embedding = embedding.join(reduced_original_embedding_df, lsuffix="_GNN", rsuffix="_Original")

In [25]:
# overwrite TSNE-1 and TSNE-2 columns with desired values
embedding["TSNE-1"] = embedding["TSNE-1_Original"]
embedding["TSNE-2"] = embedding["TSNE-2_Original"]
embedding

,TSNE-1_GNN,TSNE-2_GNN,TSNE-1_Original,TSNE-2_Original,TSNE-1,TSNE-2
DB00009,5.572844,0.555892,-4.377208,-40.772491,-4.377208,-40.772491
DB00013,34.541710,-6.758242,-5.393872,-41.004349,-5.393872,-41.004349
DB00026,38.438992,0.218512,22.908689,-37.158092,22.908689,-37.158092
DB00031,27.524046,0.308597,-4.297691,-40.854904,-4.297691,-40.854904
DB00035,-9.685918,2.996222,-8.487129,23.536123,-8.487129,23.536123
...,...,...,...,...,...,...
DB13872,1.165066,-2.759467,-0.357226,73.945618,-0.357226,73.945618
DB13878,39.012356,1.705671,-6.413913,-59.518097,-6.413913,-59.518097
DB13879,39.198452,2.789521,-6.156215,-59.767181,-6.156215,-59.767181
DB13919,-24.063969,0.624020,-2.711850,27.012836,-2.711850,27.012836


## Additional Descriptions Features

In [27]:
feature_path = "/data/giobbi/embeddings/not_aligned_with_model/drug_description_enriched.csv"
additional_description_features = pd.read_csv(feature_path, sep="\t", index_col=0)

In [28]:
embedding = embedding.join(
    additional_description_features.set_index("Drug ID"),
    how="left",
    # lsuffix='_emb',
    # rsuffix='_desc'
)


In [29]:
color_sequence = px.colors.qualitative.Set1

bins = [0, 1, 2, 4, 7, 10, 11]
count_bins = pd.cut(embedding["drug_unique_count"], bins=bins, right=False)

fig = px.scatter(
    embedding,
    x="TSNE-1",
    y="TSNE-2",
    color=embedding["pharma_class"],  # embedding['drug_unique_count'],
    color_discrete_sequence=color_sequence,
    # use hover_data to show drug IDs
    hover_data=[
        embedding.index,
        "pharma_class",
        #'Discription',
        "drug_count",
        "drug_unique_count",
    ],
    title="t-SNE Visualization of Drug Embeddings with Unique Drug Count",
    width=800,
    height=600,
)
fig.show()

## Visualize Edges


In [30]:
threshold = optimal_threshold
test_scores = vars["test_scores"]

In [31]:
data = vars["test_data"]
NUMBER_OF_EDGES = 500


In [32]:
# Select one drug and analyze its edges
most_frequent_drugs = (
    pd.Series(list(data.edge_label_index[0].cpu().numpy()) + list(data.edge_label_index[1].cpu().numpy()))
    .value_counts()
    .head(10)
)
display(most_frequent_drugs)
display(per_drug_f1)

93     21
28     21
101    21
253    17
16     15
381    15
178    15
146    14
181    14
58     13
Name: count, dtype: int64

,drug_id,n_edges,f1
index,,,
387,DB06736,13,1.0
392,DB08439,13,1.0
415,DB09068,11,1.0
152,DB00620,10,1.0
352,DB01600,10,1.0
...,...,...,...
372,DB06216,1,0.0
377,DB06401,1,0.0
382,DB06616,1,0.0


In [33]:
idx = 28
# get all positions where either source or target is idx
idcs = ((data.edge_label_index[0] == idx) | (data.edge_label_index[1] == idx)).nonzero(as_tuple=True)[0]
idcs = idcs.cpu().numpy()
idcs

array([ 26, 116, 121, 126, 185, 260, 273, 301, 315, 339, 412, 515, 560,
       600, 604, 631, 633, 636, 653, 822, 874])

In [34]:
# seed random number generator
# torch.manual_seed(44)
# idcs = torch.randperm(data.edge_label.size(0))[:NUMBER_OF_EDGES]
edge_label = data.edge_label[idcs]
edge_label_index = data.edge_label_index[:, idcs]


In [35]:
model.eval()
with torch.no_grad():
    # Step 1: Get node embeddings
    z = model.encode(data.x, data.edge_index)
    # Step 2: Predict edges (edge_label_index: shape [2, num_edges])
    logits = model.decode(z, edge_label_index).view(-1)
    probs = logits.sigmoid().cpu().numpy()  # Probabilities for each edge
    # Step 3: Apply threshold for binary prediction
    predicted_labels = (probs >= threshold).astype(int)  # Use your chosen threshold

In [36]:
# Convert the edge_label_index tensor (shape [2, num_edges]) to CPU numpy array
# then build list of (src_id, dst_id) using reversed_node_id_map.
edge_label_index_cpu = edge_label_index.cpu().numpy()  # shape (2, E)
edge_pairs = [
    (reversed_node_id_map.get(int(src)), reversed_node_id_map.get(int(dst)))
    for src, dst in edge_label_index_cpu.T  # iterate columns (E, 2)
]

In [37]:
def format_description(desc, max_length=60):
    # Insert <br> every max_length characters, or at sentence breaks
    import re

    if pd.isnull(desc):
        return ""
    # split every 20 characters
    sentences = re.findall(".{1,%d}" % max_length, desc)
    return "<br>".join(sentences)


embedding["Description_br"] = embedding["Discription"].apply(format_description)

customdata = (
    embedding[["drug_count", "pharma_class", "Description_br"]].values
    if "drug_count" in embedding.columns and "pharma_class" in embedding.columns
    else None
)

hovertemplate = (
    "Drug ID: %{text}"
    "<br>TSNE-1: %{x:.3f}"
    "<br>TSNE-2: %{y:.3f}"
    "<br>Drug Count: %{customdata[0]}"
    "<br>Pharma Class: %{customdata[1]}"
    "<br>Description: %{customdata[2]}"
    "<extra></extra>"
)

In [38]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Inputs assumed:
# embedding: pd.DataFrame with index of Drug IDs and columns ['TSNE-1', 'TSNE-2']
# edge_pairs: list[tuple[str, str]] of (src_id, dst_id) mapped to embedding.index
# edge_label: 1D tensor/array of 0/1 (ground truth) aligned with edge_pairs
# predicted_labels: 1D array of 0/1 aligned with edge_pairs

gt = np.asarray(edge_label.cpu().numpy() if hasattr(edge_label, "cpu") else edge_label).astype(int)
pred = np.asarray(predicted_labels).astype(int)
assert len(edge_pairs) == len(gt) == len(pred), "edge_pairs, gt, pred must have same length"

# Confusion categories
is_tp = (gt == 1) & (pred == 1)
is_fp = (gt == 0) & (pred == 1)
is_tn = (gt == 0) & (pred == 0)
is_fn = (gt == 1) & (pred == 0)

# Filter embedding to only nodes present in edges
nodes_in_edges = pd.Index(sorted({n for pair in edge_pairs for n in pair}))
embedding_sub = embedding.loc[embedding.index.intersection(nodes_in_edges)].copy()
embedding_sub["pharma_class"] = embedding_sub["pharma_class"].fillna("Unknown")

# Bin drug_unique_count for marker symbol
bins = [0, 1, 2, 4, 7, 10, 11]
count_bins = pd.cut(embedding_sub["drug_unique_count"], bins=bins, right=False)
symbols = ["circle", "square", "diamond", "cross", "x", "triangle-up"]
bin_to_symbol = {cat: sym for cat, sym in zip(count_bins.cat.categories, symbols)}

# Build color map for pharma_class
classes = embedding_sub["pharma_class"].astype(str).unique()
palette = px.colors.qualitative.Set1_r
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(sorted(classes))}

fig = go.Figure()

# Add node traces grouped by pharma_class and count_bin (marker symbol)
for cls in sorted(classes):
    for bin_cat in count_bins.cat.categories:
        mask = (embedding_sub["pharma_class"].astype(str) == cls) & (count_bins == bin_cat)
        df_sub = embedding_sub[mask]
        if df_sub.empty:
            continue
        node_x = df_sub["TSNE-1"].values
        node_y = df_sub["TSNE-2"].values
        node_ids = df_sub.index.astype(str).tolist()
        customdata_cls = df_sub[["drug_count", "pharma_class", "Description_br"]].values

        fig.add_trace(
            go.Scattergl(
                x=node_x,
                y=node_y,
                mode="markers",
                marker=dict(size=7, color=color_map[cls], symbol=bin_to_symbol[bin_cat]),
                text=node_ids,
                customdata=customdata_cls,
                hovertemplate=hovertemplate,
                name=f"{cls}, {bin_cat}",
            )
        )


def add_edge_lines(mask, color, name, dash="solid"):
    xs, ys = [], []
    for (src, dst), keep in zip(edge_pairs, mask):
        if not keep:
            continue
        if src not in embedding_sub.index or dst not in embedding_sub.index:
            continue
        x0, y0 = embedding_sub.loc[src, ["TSNE-1", "TSNE-2"]]
        x1, y1 = embedding_sub.loc[dst, ["TSNE-1", "TSNE-2"]]
        xs += [x0, x1, None]
        ys += [y0, y1, None]
    if xs:
        fig.add_trace(
            go.Scattergl(
                x=xs,
                y=ys,
                mode="lines",
                line=dict(color=color, width=1.0, dash=dash),
                name=name,
                hoverinfo="skip",
            )
        )


# Add edges by confusion class with requested colors/styles
add_edge_lines(is_tp, "rgba(0,180,0,0.85)", "TP (gt=1, pred=1)", dash="solid")
add_edge_lines(is_fn, "rgba(160,180,0,0.85)", "FN (gt=1, pred=0)", dash="dash")
add_edge_lines(is_tn, "rgba(220,20,60,0.85)", "TN (gt=0, pred=0)", dash="solid")
add_edge_lines(is_fp, "rgba(220,120,90,0.85)", "FP (gt=0, pred=1)", dash="dash")

fig.update_layout(
    title=f"t-SNE: TP={np.sum(is_tp)}, TN={np.sum(is_tn)}, FP={np.sum(is_fp)}, FN={np.sum(is_fn)}",
    width=950,
    height=750,
    legend=dict(itemsizing="constant"),
)
fig.show()

# print confusion matrix for this drug
drug_id = reversed_node_id_map[idx]
confusion_matrix_for_drug(drug_id, test_data, test_scores, optimal_threshold, reversed_node_id_map)

Confusion Matrix for drug ID DB00215:
[tn, fp]
[fn, tp]

[[ 4  6]
 [ 0 11]]


## All nodes

In [39]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# Fill missing pharma_class
embedding["pharma_class"] = embedding["pharma_class"].fillna("Unknown")

# Build color map for pharma_class
classes = embedding["pharma_class"].astype(str).unique()
palette = px.colors.qualitative.Set1
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(sorted(classes))}

fig = go.Figure()

# Add node traces grouped and colored by pharma_class (legend per class)
for cls in sorted(classes):
    df_cls = embedding[embedding["pharma_class"].astype(str) == cls]
    node_x = df_cls["TSNE-1"].values
    node_y = df_cls["TSNE-2"].values
    node_ids = df_cls.index.astype(str).tolist()
    customdata_cls = df_cls[["drug_count", "pharma_class", "Description_br"]].values

    fig.add_trace(
        go.Scattergl(
            x=node_x,
            y=node_y,
            mode="markers",
            marker=dict(size=7, color=color_map[cls]),
            text=node_ids,
            customdata=customdata_cls,
            hovertemplate=hovertemplate,
            name=f"{cls}",
        )
    )

# --- Edge lines by confusion class (TP, FN, TN, FP) ---
# Compute confusion categories
# gt: ground truth, pred: predicted_labels
# edge_pairs, gt, pred must be aligned

gt = np.asarray(edge_label.cpu().numpy() if hasattr(edge_label, "cpu") else edge_label).astype(int)
pred = np.asarray(predicted_labels).astype(int)
assert len(edge_pairs) == len(gt) == len(pred), "edge_pairs, gt, pred must have same length"

is_tp = (gt == 1) & (pred == 1)
is_fp = (gt == 0) & (pred == 1)
is_tn = (gt == 0) & (pred == 0)
is_fn = (gt == 1) & (pred == 0)


def add_edge_lines(mask, color, name, dash="solid"):
    xs, ys = [], []
    for (src, dst), keep in zip(edge_pairs, mask):
        if not keep:
            continue
        if src not in embedding.index or dst not in embedding.index:
            continue
        x0, y0 = embedding.loc[src, ["TSNE-1", "TSNE-2"]]
        x1, y1 = embedding.loc[dst, ["TSNE-1", "TSNE-2"]]
        xs += [x0, x1, None]
        ys += [y0, y1, None]
    if xs:
        fig.add_trace(
            go.Scattergl(
                x=xs,
                y=ys,
                mode="lines",
                line=dict(color=color, width=1.0, dash=dash),
                name=name,
                hoverinfo="skip",
            )
        )


# Add edges by confusion class with requested colors/styles
add_edge_lines(is_tp, "rgba(0,180,0,0.85)", "TP (gt=1, pred=1)", dash="solid")
add_edge_lines(is_fn, "rgba(160,180,0,0.85)", "FN (gt=1, pred=0)", dash="dash")
add_edge_lines(is_tn, "rgba(220,20,60,0.85)", "TN (gt=0, pred=0)", dash="solid")
add_edge_lines(is_fp, "rgba(220,120,90,0.85)", "FP (gt=0, pred=1)", dash="dash")

fig.update_layout(
    title=f"t-SNE: TP={np.sum(is_tp)}, TN={np.sum(is_tn)}, FP={np.sum(is_fp)}, FN={np.sum(is_fn)}",
    width=950,
    height=750,
    legend=dict(itemsizing="constant"),
)
fig.show()


In [40]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch

# Ground truth positive labels. all others are negative (external Drug IDs)
positive_edges = vars["data"].edge_index
positive_edge_drug_ids = [
    (reversed_node_id_map.get(int(src)), reversed_node_id_map.get(int(dst)))
    for src, dst in positive_edges.t().cpu().numpy()
]


def plot_all_possible_edges_for_node_with_gt(
    node_id_str,
    threshold=0.544,
    limit=None,  # optionally restrict number of candidate edges for speed
):
    # Preconditions
    assert "TSNE-1" in embedding.columns and "TSNE-2" in embedding.columns, (
        "Embedding must have TSNE-1/TSNE-2"
    )
    assert isinstance(node_id_map, dict) and isinstance(reversed_node_id_map, dict), (
        "node_id_map and reversed_node_id_map must be dicts"
    )
    if node_id_str not in node_id_map:
        raise ValueError(f"Node '{node_id_str}' not found in node_id_map")
    node_idx = node_id_map[node_id_str]

    # Build undirected-normalized GT set from external Drug IDs
    # Normalize pair as (min_id, max_id) strings to handle undirected membership checks robustly
    def norm_pair_ext(a, b):
        return (a, b) if a <= b else (b, a)

    gt_pos_ext = set(
        norm_pair_ext(str(a), str(b)) for a, b in positive_edge_drug_ids if a is not None and b is not None
    )

    # Full graph references
    full_data = graph_with_emb
    N = full_data.x.shape[0]
    device = next(model.parameters()).device if hasattr(model, "parameters") else "cpu"

    # Candidate edges from chosen node to all others (excluding self)
    all_targets = np.arange(N, dtype=np.int64)
    all_targets = all_targets[all_targets != node_idx]
    if limit is not None and all_targets.size > limit:
        all_targets = all_targets[:limit]

    # Internal indices for decode
    srcs = np.full(all_targets.shape, node_idx, dtype=np.int64)
    dsts = all_targets
    edge_index_candidate = torch.tensor([srcs, dsts], dtype=torch.long, device=device)

    # External Drug IDs for GT matching and plotting
    edge_pairs_external = [
        (reversed_node_id_map.get(int(s)), reversed_node_id_map.get(int(d))) for s, d in zip(srcs, dsts)
    ]
    # Normalize external pairs for GT membership check
    edge_pairs_external_norm = [norm_pair_ext(str(a), str(b)) for a, b in edge_pairs_external]

    # Ground-truth labels from GT set of external IDs
    gt = np.array([1 if pair in gt_pos_ext else 0 for pair in edge_pairs_external_norm], dtype=np.int32)

    # Predict probabilities
    model.eval()
    with torch.no_grad():
        z = model.encode(full_data.x.to(device), full_data.edge_index.to(device))
        logits = model.decode(z, edge_index_candidate).view(-1)
        probs = logits.sigmoid().detach().cpu().numpy()
    pred = (probs >= threshold).astype(np.int32)

    # Confusion masks
    assert len(gt) == len(pred) == len(edge_pairs_external)
    is_tp = (gt == 1) & (pred == 1)
    is_fp = (gt == 0) & (pred == 1)
    is_tn = (gt == 0) & (pred == 0)
    is_fn = (gt == 1) & (pred == 0)

    # Node scatter: color by pharma_class
    emb = embedding.copy()
    emb["pharma_class"] = emb["pharma_class"].fillna("Unknown")
    classes = emb["pharma_class"].astype(str).unique()
    palette = px.colors.qualitative.Set1_r
    color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(sorted(classes))}

    fig = go.Figure()
    for cls in sorted(classes):
        df_cls = emb[emb["pharma_class"].astype(str) == cls]
        fig.add_trace(
            go.Scattergl(
                x=df_cls["TSNE-1"].values,
                y=df_cls["TSNE-2"].values,
                mode="markers",
                marker=dict(size=7, color=color_map[cls]),
                text=df_cls.index.astype(str).tolist(),
                customdata=df_cls[["drug_count", "pharma_class", "Description_br"]].values
                if set(["drug_count", "pharma_class", "Description_br"]).issubset(df_cls.columns)
                else None,
                hovertemplate=(
                    "Drug ID: %{text}"
                    "<br>TSNE-1: %{x:.3f}"
                    "<br>TSNE-2: %{y:.3f}"
                    "<br>Drug Count: %{customdata[0]}"
                    "<br>Pharma Class: %{customdata[1]}"
                    "<br>Description: %{customdata[2]}"
                    "<extra></extra>"
                )
                if set(["drug_count", "pharma_class", "Description_br"]).issubset(df_cls.columns)
                else None,
                name=f"{cls}",
            )
        )

    # Edge plotting by confusion class
    def add_edge_lines(mask, color, name, dash="solid"):
        xs, ys = [], []
        for (src_ext, dst_ext), keep in zip(edge_pairs_external, mask):
            if not keep:
                continue
            if (src_ext not in emb.index) or (dst_ext not in emb.index):
                continue
            x0, y0 = emb.loc[src_ext, ["TSNE-1", "TSNE-2"]]
            x1, y1 = emb.loc[dst_ext, ["TSNE-1", "TSNE-2"]]
            xs += [x0, x1, None]
            ys += [y0, y1, None]
        if xs:
            fig.add_trace(
                go.Scattergl(
                    x=xs,
                    y=ys,
                    mode="lines",
                    line=dict(color=color, width=1.2, dash=dash),
                    name=name,
                    hoverinfo="skip",
                )
            )

    add_edge_lines(is_tp, "rgba(0,180,0,0.95)", f"TP edges of {node_id_str}", dash="solid")
    # add_edge_lines(is_fn, 'rgba(160,180,0,0.95)', f"FN edges of {node_id_str}", dash='dash')
    # add_edge_lines(is_tn, 'rgba(220,20,60,0.95)', f"TN edges of {node_id_str}", dash='solid')
    add_edge_lines(is_fp, "rgba(220,120,90,0.95)", f"FP edges of {node_id_str}", dash="dash")

    # Highlight focus node
    if node_id_str in emb.index:
        fig.add_trace(
            go.Scattergl(
                x=[emb.loc[node_id_str, "TSNE-1"]],
                y=[emb.loc[node_id_str, "TSNE-2"]],
                mode="markers",
                marker=dict(size=12, color="gold", symbol="star"),
                name=f"Focus: {node_id_str}",
                hoverinfo="skip",
            )
        )

    summary_text = (
        f"TP={is_tp.sum()}, FP={is_fp.sum()}, TN={is_tn.sum()}, FN={is_fn.sum()} (threshold={threshold:.3f})"
    )
    fig.update_layout(
        title=f"All candidate edges around '{node_id_str}' — {summary_text}",
        width=950,
        height=750,
        legend=dict(itemsizing="constant"),
    )
    fig.show()


# Example:
# plot_all_possible_edges_for_node_with_gt('DB00331', threshold=0.544)

In [41]:
plot_all_possible_edges_for_node_with_gt("DB00331", threshold=0.997)

ValueError: Node 'DB00331' not found in node_id_map

# Not used anymore

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Inputs assumed:
# embedding: pd.DataFrame with index of Drug IDs and columns ['TSNE-1', 'TSNE-2']
# edge_pairs: list[tuple[str, str]] of (src_id, dst_id) mapped to embedding.index
# edge_label: 1D tensor/array of 0/1 (ground truth) aligned with edge_pairs
# predicted_labels: 1D array of 0/1 aligned with edge_pairs

gt = np.asarray(edge_label.cpu().numpy() if hasattr(edge_label, "cpu") else edge_label).astype(int)
pred = np.asarray(predicted_labels).astype(int)
assert len(edge_pairs) == len(gt) == len(pred), "edge_pairs, gt, pred must have same length"

# Confusion categories
is_tp = (gt == 1) & (pred == 1)
is_fp = (gt == 0) & (pred == 1)
is_tn = (gt == 0) & (pred == 0)
is_fn = (gt == 1) & (pred == 0)

# Filter embedding to only nodes present in edges
nodes_in_edges = pd.Index(sorted({n for pair in edge_pairs for n in pair}))
embedding_sub = embedding.loc[embedding.index.intersection(nodes_in_edges)].copy()


embedding_sub["pharma_class"] = embedding_sub["pharma_class"].fillna("Unknown")

# Build color map for pharma_class
classes = embedding_sub["pharma_class"].astype(str).unique()
palette = px.colors.qualitative.Set1_r
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(sorted(classes))}

fig = go.Figure()

# Add node traces grouped and colored by pharma_class (legend per class)
for cls in sorted(classes):
    df_cls = embedding_sub[embedding_sub["pharma_class"].astype(str) == cls]
    node_x = df_cls["TSNE-1"].values
    node_y = df_cls["TSNE-2"].values
    node_ids = df_cls.index.astype(str).tolist()
    customdata_cls = df_cls[["drug_count", "pharma_class", "Description_br"]].values

    fig.add_trace(
        go.Scattergl(
            x=node_x,
            y=node_y,
            mode="markers",
            marker=dict(size=7, color=color_map[cls]),
            text=node_ids,
            customdata=customdata_cls,
            hovertemplate=hovertemplate,
            name=f"{cls}",
        )
    )


def add_edge_lines(mask, color, name, dash="solid"):
    xs, ys = [], []
    for (src, dst), keep in zip(edge_pairs, mask):
        if not keep:
            continue
        if src not in embedding_sub.index or dst not in embedding_sub.index:
            continue
        x0, y0 = embedding_sub.loc[src, ["TSNE-1", "TSNE-2"]]
        x1, y1 = embedding_sub.loc[dst, ["TSNE-1", "TSNE-2"]]
        xs += [x0, x1, None]
        ys += [y0, y1, None]
    if xs:
        fig.add_trace(
            go.Scattergl(
                x=xs,
                y=ys,
                mode="lines",
                line=dict(color=color, width=1.0, dash=dash),
                name=name,
                hoverinfo="skip",
            )
        )


# Add edges by confusion class with requested colors/styles
add_edge_lines(is_tp, "rgba(0,180,0,0.85)", "TP (gt=1, pred=1)", dash="solid")
add_edge_lines(is_fn, "rgba(160,180,0,0.85)", "FN (gt=1, pred=0)", dash="dash")
add_edge_lines(is_tn, "rgba(220,20,60,0.85)", "TN (gt=0, pred=0)", dash="solid")
add_edge_lines(is_fp, "rgba(220,120,90,0.85)", "FP (gt=0, pred=1)", dash="dash")

fig.update_layout(
    title="t-SNE: Nodes colored by pharma_class; TP (green), FN (green dashed), TN (red), FP (red dashed)",
    width=950,
    height=750,
    legend=dict(itemsizing="constant"),
)
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Assume:
# - embedding: DataFrame with index = Drug IDs, columns ['TSNE-1', 'TSNE-2']
# - edge_pairs: list of (src_id, dst_id) using reversed_node_id_map (strings/IDs in embedding.index)
# - predicted_labels: np.array of 0/1 for each edge in edge_pairs
# - probs: np.array of probabilities for each edge in edge_pairs

# 1) Base scatter of nodes
node_x = embedding["TSNE-1"].values
node_y = embedding["TSNE-2"].values
node_ids = embedding.index.astype(str).tolist()


fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=node_x,
        y=node_y,
        mode="markers",
        marker=dict(size=6, color="rgba(0,0,150,0.6)"),
        text=node_ids,
        customdata=customdata,
        hovertemplate=hovertemplate,
        name="Drugs",
    )
)

# 2) Edge lines: draw as separate traces for positives vs negatives
# Build arrays of line segments for efficiency
pos_x, pos_y = [], []
neg_x, neg_y = [], []

# Optional: color edges by probability using a continuous colormap
use_continuous_color = False

for (src, dst), pl, p in zip(edge_pairs, predicted_labels, probs):
    # Skip edges if either endpoint missing from embedding
    if src not in embedding.index or dst not in embedding.index:
        continue
    x0, y0 = embedding.loc[src, ["TSNE-1", "TSNE-2"]]
    x1, y1 = embedding.loc[dst, ["TSNE-1", "TSNE-2"]]
    if use_continuous_color:
        # Add as individual trace colored by prob (more flexible but heavier for many edges)
        fig.add_trace(
            go.Scattergl(
                x=[x0, x1],
                y=[y0, y1],
                mode="lines",
                line=dict(width=1.5, color=px.colors.sample_colorscale("Viridis", [p])[0]),
                hovertemplate=f"Edge: {src} — {dst}<br>Prob: {p:.3f}<extra></extra>",
                showlegend=False,
            )
        )
    else:
        if pl == 1:
            pos_x += [x0, x1, None]
            pos_y += [y0, y1, None]
        else:
            neg_x += [x0, x1, None]
            neg_y += [y0, y1, None]

if not use_continuous_color:
    # Positive edges
    fig.add_trace(
        go.Scattergl(
            x=pos_x,
            y=pos_y,
            mode="lines",
            line=dict(color="rgba(20,180,60,0.6)", width=1.5),  # crimson 'rgba(220,20,60,0.6)'
            name="Positive predicted edges",
            hoverinfo="skip",
        )
    )
    # Negative edges
    fig.add_trace(
        go.Scattergl(
            x=neg_x,
            y=neg_y,
            mode="lines",
            line=dict(color="rgba(220,20,60,0.85)", width=1.0),
            name="Negative predicted edges",
            hoverinfo="skip",
        )
    )

fig.update_layout(
    title="t-SNE of Drug Embeddings with Edges", width=900, height=700, legend=dict(itemsizing="constant")
)
fig.show()